> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# Llama・Gemma：特異値ヒストグラムによるMP適合診断

保存済みスペクトルを用いて、以下を可視化します。**モデル取得・SVD・GPU・APIキーは不要**です。

1. DE前後で `fullKS`、上端のみの `bulkKS`、両端の `bulkKS` が大きく変わる行列と、変化が小さい行列。
2. MLPの `up/down/gate` とAttentionの `q/k/v/o` を種類別に比較。
3. 全体KSが大きい一方で両端bulkKSが小さい行列。
4. MPバルク外の割合が大きい行列について、全体・上端除外・両端除外の各正規化をMP密度と比較。
5. 従来のLRA順序に使った `KS_postDE_1` が大きい行列について、preDE/postDEのスペクトルを比較。

既定の横軸はMP尺度で正規化した特異値 $s/\sqrt{n\sigma^2}=\sqrt{\lambda/\sigma^2}$ です。これによりDE前後で尺度が変わっても形を比較できます。`AXIS_MODE='raw'` にすると元の特異値 $s=\sqrt{n\lambda}$ を表示します。

ヒストグラムは各パネルで残された標本だけを総質量1に正規化します。これはKS計算時の経験CDF正規化と同じです。MP曲線も面積1です。閾値外を除いてKSが下がることは、閾値の理論的妥当性や独立ノイズ性の証明ではありません。

In [ ]:
# 初回だけ不足ライブラリをインストールします。
import importlib.util, subprocess, sys
packages = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib'}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

## 設定

`SELECTION_METRIC` は代表行列の選択に使うKSです。既定は両端bulkKSですが、`fullKS` または `upper_only_bulkKS` に変更して再実行できます。

In [ ]:
import io, re, json, hashlib
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if IN_COLAB:
    DRIVE_DATA = Path('/content/drive/MyDrive/TUS/hashiguchi/data')
else:
    choices = list((Path.home()/'Library/CloudStorage').glob('GoogleDrive-*/*/TUS/hashiguchi/data'))
    if len(choices) != 1:
        raise RuntimeError('DRIVE_DATAを自分のDriveのdataフォルダに設定してください')
    DRIVE_DATA = choices[0]

SOURCE_TAG = 'bulk-ks-20260925-v1-full'
TWO_SIDED_TAG = 'two-sided-bulk-ks-v1'
OUTPUT_TAG = 'spectral-histograms-v1'
OUTPUT = DRIVE_DATA/'bulk_ks_visualization'/OUTPUT_TAG
OUTPUT.mkdir(parents=True, exist_ok=True)

AXIS_MODE = 'mp_scaled'       # 'mp_scaled' または 'raw'
SELECTION_METRIC = 'two_sided_bulkKS'  # fullKS / upper_only_bulkKS / two_sided_bulkKS
BINS = 70
DISPLAY_QUANTILE = 0.995      # 極端な外れ値でバルクが潰れないよう表示範囲だけ制限
OUTLIER_TOP_PER_FAMILY = 2    # 各モデル×Attention/MLPで詳細表示する件数
KS_POSTDE_1_TOP_N = 5         # 従来のLRA順序に使ったKS_postDE_1の上位件数（モデルごと）
AUTO_DISCONNECT = True

MODULE_ORDER = ['q_proj','k_proj','v_proj','o_proj','up_proj','gate_proj','down_proj']
FAMILY = {m:('Attention' if m in {'q_proj','k_proj','v_proj','o_proj'} else 'MLP') for m in MODULE_ORDER}
print('Input:', DRIVE_DATA)
print('Output:', OUTPUT)

## データ読み込みと整合性確認

両端bulkKS結果を主表とし、元実験CSVからスペクトルファイルを特定します。可視化対象になったファイルだけを読みます。

In [ ]:
result_path = DRIVE_DATA/'bulk_ks_two_sided'/TWO_SIDED_TAG/'two_sided_bulk_ks.csv'
results = pd.read_csv(result_path)
primary = results[results.method.eq('matched_upper')].copy()
assert len(primary) == 434*3
assert not primary.duplicated(['model','name','condition']).any()
assert set(primary.condition) == {'preDE','postDE','DE_1'}
assert np.isfinite(primary[['fullKS','upper_only_bulkKS','two_sided_bulkKS']]).all().all()

source_rows = []
source_dirs = {}
for model, expected in {'llama':196, 'gemma':238}.items():
    directory = DRIVE_DATA/'bulk_ks'/model/SOURCE_TAG
    frame = pd.read_csv(directory/'bulk_ks.csv')
    assert len(frame) == expected and not frame.name.duplicated().any()
    frame = frame.assign(model=model, family=frame.module_type.map(FAMILY))
    source_rows.append(frame)
    source_dirs[model] = directory
source = pd.concat(source_rows, ignore_index=True)
assert set(source[['model','name']].itertuples(index=False, name=None)) == set(primary[['model','name']].itertuples(index=False, name=None))

post = primary[primary.condition.eq('postDE')].copy()
post['outlier_ratio'] = post.low_ratio + post.high_ratio
print(f'{len(primary):,} diagnostic rows / {len(source):,} matrices loaded')

## DE変化の全体像

差は `postDE − preDE` です。負値ほどDE後にKSが小さくなっています。まずモジュール種類別の分布と、代表行列の選択表を確認します。

In [ ]:
metrics = ['fullKS','upper_only_bulkKS','two_sided_bulkKS']
wide = primary[primary.condition.isin(['preDE','postDE'])].pivot(
    index=['model','name','module_type','block_index'], columns='condition', values=metrics)
wide.columns = [f'{metric}_{condition}' for metric,condition in wide.columns]
wide = wide.reset_index()
for metric in metrics:
    wide[f'delta_{metric}'] = wide[f'{metric}_postDE'] - wide[f'{metric}_preDE']
wide['family'] = wide.module_type.map(FAMILY)

long_delta = wide.melt(
    id_vars=['model','name','module_type','family'],
    value_vars=[f'delta_{m}' for m in metrics],
    var_name='metric', value_name='delta')
long_delta['metric'] = long_delta.metric.str.removeprefix('delta_')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, layout='constrained')
colors = {'fullKS':'#4c78a8','upper_only_bulkKS':'#f58518','two_sided_bulkKS':'#54a24b'}
positions = np.arange(len(MODULE_ORDER))
width = .22
for ax, model in zip(axes, ['llama','gemma']):
    for j, metric in enumerate(metrics):
        vals = [long_delta.query('model == @model and module_type == @module and metric == @metric').delta.values
                for module in MODULE_ORDER]
        bp = ax.boxplot(vals, positions=positions+(j-1)*width, widths=width*.9,
                        patch_artist=True, showfliers=False, medianprops={'color':'black'})
        for box in bp['boxes']:
            box.set_facecolor(colors[metric]); box.set_alpha(.72)
    ax.axhline(0, color='0.35', lw=1)
    ax.set_title(f'{model}: postDE - preDE (negative = improvement)')
    ax.set_ylabel('KS difference')
    ax.grid(axis='y', alpha=.2)
axes[-1].set_xticks(positions, MODULE_ORDER)
handles = [plt.Line2D([0],[0], color=colors[m], lw=7, label=m) for m in metrics]
axes[0].legend(handles=handles, loc='upper left', ncol=3, fontsize=9)
fig.savefig(OUTPUT/'01_de_delta_by_module.png', dpi=160)
plt.show()

selection_rows = []
delta_col = f'delta_{SELECTION_METRIC}'
for (model,module), group in wide.groupby(['model','module_type'], sort=False):
    improved = group.loc[group[delta_col].idxmin()].copy()
    stable = group.loc[group[delta_col].abs().idxmin()].copy()
    for label,row in [('largest_decrease',improved),('smallest_absolute_change',stable)]:
        selection_rows.append({**row.to_dict(),'selection':label,'selection_metric':SELECTION_METRIC})
selected_de = pd.DataFrame(selection_rows)
display(selected_de[['model','module_type','selection','name',delta_col,
                     'delta_fullKS','delta_upper_only_bulkKS','delta_two_sided_bulkKS']]
        .sort_values(['model','module_type','selection']))

## 描画関数

青は全特異値、橙は上端より大きい値を除外した集合、緑は上下端の両方を除外した集合です。各集合を個別に総質量1へ正規化し、同じ分散・アスペクト比のMP密度を重ねます。破線は除外閾値です。

In [ ]:
def safe_name(text):
    return re.sub(r'[^A-Za-z0-9_.-]+','_',str(text))

@lru_cache(maxsize=64)
def load_spectrum(model, name):
    row = source[(source.model==model)&(source.name==name)].iloc[0]
    path = (source_dirs[model]/row.spectrum_file).resolve()
    if not path.is_relative_to(source_dirs[model].resolve()):
        raise ValueError('Invalid spectrum path')
    payload = path.read_bytes()
    with np.load(io.BytesIO(payload), allow_pickle=False) as data:
        if str(data['layer_name'].item()) != name:
            raise ValueError('Spectrum identity mismatch')
        return {'preDE':np.asarray(data['pre'],float),
                'postDE':np.asarray(data['post'],float),
                'p':int(data['p']), 'n':int(data['n']),
                'sha256':hashlib.sha256(payload).hexdigest()}

def singular_axis(eigenvalues, record):
    values = np.asarray(eigenvalues,float)
    if AXIS_MODE == 'mp_scaled':
        return np.sqrt(np.maximum(values/record.sigma2,0))
    if AXIS_MODE == 'raw':
        return np.sqrt(np.maximum(values*record.n,0))
    raise ValueError("AXIS_MODE must be 'mp_scaled' or 'raw'")

def threshold_axis(value, record):
    if AXIS_MODE == 'mp_scaled':
        return np.sqrt(max(value/record.sigma2,0))
    return np.sqrt(max(value*record.n,0))

def mp_singular_density(record, x):
    # λ=s²/n。変数変換 f_s(s)=f_λ(s²/n)·2s/n を用いる。
    gamma = record.gamma
    if AXIS_MODE == 'mp_scaled':
        t = x
        z = t*t
        a,b = (1-np.sqrt(gamma))**2, (1+np.sqrt(gamma))**2
        fz = np.sqrt(np.maximum((b-z)*(z-a),0))/(2*np.pi*gamma*np.maximum(z,np.finfo(float).tiny))
        return 2*t*fz
    s = x
    lam = s*s/record.n
    a,b = record.sigma2*(1-np.sqrt(gamma))**2, record.sigma2*(1+np.sqrt(gamma))**2
    fl = np.sqrt(np.maximum((b-lam)*(lam-a),0))/(2*np.pi*gamma*record.sigma2*np.maximum(lam,np.finfo(float).tiny))
    return fl*2*s/record.n

def record_for(model,name,condition):
    rows = primary[(primary.model==model)&(primary.name==name)&(primary.condition==condition)]
    if len(rows) != 1: raise ValueError('Diagnostic row lookup failed')
    return rows.iloc[0]

def subsets(eigenvalues, record):
    values = np.asarray(eigenvalues,float)
    return {
        'Full': values,
        'Upper-only bulk': values[values <= record.upper_threshold],
        'Two-sided bulk': values[(values >= record.lower_threshold)&(values <= record.upper_threshold)]
    }

def plot_distribution(ax, eigenvalues, record, modes=('Full','Upper-only bulk','Two-sided bulk'), title=''):
    sets = subsets(eigenvalues,record)
    palette = {'Full':'#4c78a8','Upper-only bulk':'#f58518','Two-sided bulk':'#54a24b'}
    styles = {'Full':'-','Upper-only bulk':'--','Two-sided bulk':'-.'}
    all_x = singular_axis(eigenvalues,record)
    high = threshold_axis(record.upper_threshold,record)
    lo = threshold_axis(record.lower_threshold,record)
    mp_hi = threshold_axis(record.mp_upper,record)
    xmax = max(np.quantile(all_x,DISPLAY_QUANTILE), high*1.08, mp_hi*1.12)
    xmin = 0
    bins = np.linspace(xmin,xmax,BINS+1)
    for mode in modes:
        x = singular_axis(sets[mode],record)
        if len(x):
            ax.hist(x, bins=bins, density=True, histtype='step', lw=1.5,
                    color=palette[mode], linestyle=styles[mode], label=f'{mode} (n={len(x)})')
    grid = np.linspace(max(xmax/2000,1e-8),xmax,1200)
    ax.plot(grid,mp_singular_density(record,grid),color='black',lw=1.8,label='MP density')
    if lo > 0: ax.axvline(lo,color='#54a24b',lw=1,ls=':')
    ax.axvline(high,color='#f58518',lw=1,ls=':')
    clipped = int((all_x>xmax).sum())
    ax.set_xlim(xmin,xmax); ax.set_title(title)
    ax.set_xlabel(r'$s/\sqrt{n\sigma^2}$' if AXIS_MODE=='mp_scaled' else 'singular value s')
    ax.set_ylabel('density')
    ax.grid(axis='y',alpha=.18)
    stats=(f"full={record.fullKS:.4f}  upper={record.upper_only_bulkKS:.4f}  two={record.two_sided_bulkKS:.4f}\n"
           f"low={record.low_ratio:.1%}  high={record.high_ratio:.1%}  bulk={record.bulk_ratio:.1%}")
    if clipped: stats += f'  clipped={clipped}'
    ax.text(.99,.97,stats,transform=ax.transAxes,ha='right',va='top',fontsize=8,
            bbox={'facecolor':'white','alpha':.78,'edgecolor':'none'})

def compare_pre_post(model,name,selection_label=None):
    spectrum=load_spectrum(model,name)
    fig,axes=plt.subplots(1,2,figsize=(14,4.8),layout='constrained')
    for ax,condition in zip(axes,['preDE','postDE']):
        rec=record_for(model,name,condition)
        plot_distribution(ax,spectrum[condition],rec,title=condition)
    axes[0].legend(fontsize=8,loc='upper left')
    module=record_for(model,name,'postDE').module_type
    label=f' — {selection_label}' if selection_label else ''
    fig.suptitle(f'{model} / {module} / {name}{label}')
    return fig


def normalization_pre_post_grid(model,name,selection_note='postDE-selected contrast'):
    # postDEで抽出した同一行列について、preDE/postDEを同じ構成で比較する。
    spectrum=load_spectrum(model,name)
    modes=('Full','Upper-only bulk','Two-sided bulk')
    fig,axes=plt.subplots(2,3,figsize=(16,8.5),layout='constrained')
    for row_i,condition in enumerate(['preDE','postDE']):
        rec=record_for(model,name,condition)
        for col_i,mode in enumerate(modes):
            plot_distribution(axes[row_i,col_i],spectrum[condition],rec,
                              modes=(mode,),title=f'{condition} / {mode}')
        axes[row_i,0].legend(fontsize=8,loc='upper left')
    module=record_for(model,name,'postDE').module_type
    fig.suptitle(f'{model} / {module} / {name} — {selection_note}, pre/post comparison')
    return fig

def normalization_triptych(model,name,condition='postDE'):
    spectrum=load_spectrum(model,name)
    rec=record_for(model,name,condition)
    sets=subsets(spectrum[condition],rec)
    fig,axes=plt.subplots(1,3,figsize=(16,4.5),layout='constrained')
    for ax,mode in zip(axes,sets):
        plot_distribution(ax,spectrum[condition],rec,modes=(mode,),title=mode)
    axes[0].legend(fontsize=8,loc='upper left')
    fig.suptitle(f'{model} / {rec.module_type} / {name} / {condition} — normalization comparison')
    return fig

## モジュール別：DEで大きく改善した例と変化が小さい例

各モデル・各モジュール種類について、設定したKS指標の減少が最大の行列と、絶対変化が最小の行列を並べます。各図の左右がDE前後です。

In [ ]:
for model in ['llama','gemma']:
    for module in MODULE_ORDER:
        chosen = selected_de[(selected_de.model==model)&(selected_de.module_type==module)]
        fig, axes = plt.subplots(2,2,figsize=(14,9),layout='constrained')
        for row_i,(_,item) in enumerate(chosen.sort_values('selection').iterrows()):
            spectrum=load_spectrum(model,item['name'])
            for col_i,condition in enumerate(['preDE','postDE']):
                rec=record_for(model,item['name'],condition)
                plot_distribution(axes[row_i,col_i],spectrum[condition],rec,
                                  title=f"{item['selection']} / {condition}")
            axes[row_i,0].set_ylabel(axes[row_i,0].get_ylabel()+f"\n{item['name']}")
        axes[0,0].legend(fontsize=8,loc='upper left')
        fig.suptitle(f'{model} / {FAMILY[module]} / {module} — selected by {SELECTION_METRIC}')
        path=OUTPUT/f'02_de_examples_{model}_{module}.png'
        fig.savefig(path,dpi=150); plt.show(); plt.close(fig)

## 全体KSが大きく両端bulkKSが小さい行列

postDEで `fullKS > 0.08` かつ `two-sided bulkKS < 0.05` の行列を候補として抽出します。抽出条件はpostDEだけに適用し、同一行列のpreDEとpostDEを上下2行、全体・上端除外・両端除外を3列で並べます。厳密条件 `fullKS > 0.1` かつ `two-sided bulkKS < 0.05` も表に示します。

In [ ]:
contrast = post[(post.fullKS>.08)&(post.two_sided_bulkKS<.05)].copy()
contrast['full_minus_two'] = contrast.fullKS-contrast.two_sided_bulkKS
contrast['strict_0.1_0.05'] = (contrast.fullKS>.1)&(contrast.two_sided_bulkKS<.05)
contrast = contrast.sort_values('full_minus_two',ascending=False)
display(contrast[['model','name','module_type','fullKS','upper_only_bulkKS','two_sided_bulkKS',
                  'full_minus_two','low_ratio','high_ratio','strict_0.1_0.05']])
contrast.to_csv(OUTPUT/'full_large_bulk_small_candidates.csv',index=False)

for _,row in contrast.head(8).iterrows():
    fig=normalization_pre_post_grid(row.model,row['name'])
    fig.savefig(OUTPUT/f"03_contrast_prepost_{row.model}_{safe_name(row['name'])}.png",dpi=150)
    plt.show(); plt.close(fig)

## 従来のLRA順序：KS_postDE_1が大きい行列

従来のLRA実験で優先順位に使った `KS_postDE_1`（DE後スペクトルを分散1のMP分布と比較した全体KS）が大きい行列を、LLaMA・Gemmaそれぞれ上位5件抽出します。抽出した同一行列について、preDEとpostDEを上下2行、全体・上端除外・両端除外を3列で表示します。図中のKSと閾値は各行の条件に対応する値であり、選択順位を決めた `KS_postDE_1` とは区別して表示します。

In [ ]:
ks_postde1 = primary[primary.condition.eq('DE_1')].copy()
ks_postde1['KS_postDE_1'] = ks_postde1['fullKS']
ks_postde1['KS_postDE_1_rank'] = (ks_postde1.groupby('model')['KS_postDE_1']
                                      .rank(method='first',ascending=False).astype(int))
ks_postde1_top = (ks_postde1[ks_postde1.KS_postDE_1_rank <= KS_POSTDE_1_TOP_N]
                  .sort_values(['model','KS_postDE_1_rank']))
assert ks_postde1_top.groupby('model').size().eq(KS_POSTDE_1_TOP_N).all()
# 元実験CSVのKS_postDE_1と一致することを確認してから順位を可視化する。
ks_check = ks_postde1_top.merge(source[['model','name','KS_postDE_1']],
                                  on=['model','name'],suffixes=('_current','_source'))
assert np.allclose(ks_check.KS_postDE_1_current,ks_check.KS_postDE_1_source,rtol=0,atol=2e-7)
display(ks_postde1_top[['model','KS_postDE_1_rank','name','module_type','KS_postDE_1',
                         'upper_only_bulkKS','two_sided_bulkKS','low_ratio','high_ratio']])
ks_postde1_top.to_csv(OUTPUT/f'ks_postDE_1_top{KS_POSTDE_1_TOP_N}.csv',index=False)

for _,row in ks_postde1_top.iterrows():
    rank=int(row.KS_postDE_1_rank)
    fig=normalization_pre_post_grid(row.model,row['name'],selection_note=f'KS_postDE_1 rank {rank}')
    fig.savefig(OUTPUT/f"05_ks_postDE_1_rank{rank}_{row.model}_{safe_name(row['name'])}.png",dpi=150)
    plt.show(); plt.close(fig)

## 外れ値割合が大きい行列

AttentionとMLPを分け、各モデル・各familyでDE後の上下端外れ値率が最大の行列を表示します。3パネルはそれぞれの標本集合で密度を総質量1へ正規化しています。

In [ ]:
heavy = (post.assign(family=post.module_type.map(FAMILY))
         .sort_values('outlier_ratio',ascending=False)
         .groupby(['model','family'],sort=False).head(OUTLIER_TOP_PER_FAMILY)
         .sort_values(['model','family','outlier_ratio'],ascending=[True,True,False]))
display(heavy[['model','family','name','module_type','fullKS','upper_only_bulkKS','two_sided_bulkKS',
               'low_ratio','high_ratio','outlier_ratio','bulk_ratio']])
heavy.to_csv(OUTPUT/'high_outlier_ratio_candidates.csv',index=False)

for _,row in heavy.iterrows():
    fig=normalization_triptych(row.model,row['name'],'postDE')
    fig.savefig(OUTPUT/f"04_outlier_heavy_{row.model}_{safe_name(row['name'])}.png",dpi=150)
    plt.show(); plt.close(fig)

## 確認用一覧と解釈上の注意

- `fullKS`、上端のみbulkKS、両端bulkKSは全図に併記しています。
- 横軸の表示範囲外にある極端な特異値の個数は `clipped` として表示します。KS計算や外れ値率から除外しているわけではありません。
- subsetごとの正規化により形状を比較できますが、除外数が多いほどKSが下がりやすくなる選択効果があります。
- `0.05`はKS距離の探索上の目安であり、p値や有意水準ではありません。
- LRAのPPL維持や、外れ値が機能的な信号であることはこの図だけでは判断できません。

In [ ]:
manifest = {
    'source_results': str(result_path), 'source_tag': SOURCE_TAG,
    'two_sided_tag': TWO_SIDED_TAG, 'axis_mode': AXIS_MODE,
    'selection_metric': SELECTION_METRIC, 'bins': BINS,
    'display_quantile': DISPLAY_QUANTILE,
    'selected_de_rows': len(selected_de), 'contrast_rows': len(contrast),
    'ks_postDE_1_top_rows': len(ks_postde1_top), 'ks_postDE_1_top_n_per_model': KS_POSTDE_1_TOP_N,
    'heavy_rows': len(heavy), 'generated_utc': pd.Timestamp.utcnow().isoformat()
}
(OUTPUT/'visualization_metadata.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
expected = 1 + 2*len(MODULE_ORDER) + min(8,len(contrast)) + len(ks_postde1_top) + len(heavy)
pngs = list(OUTPUT.glob('*.png'))
assert len(pngs) >= expected, (len(pngs),expected)
print(f'Complete: {len(pngs)} PNG files saved to {OUTPUT}')

In [ ]:
# 全保存の成功後だけColabとの接続を切ります。ローカル実行では何もしません。
if IN_COLAB and AUTO_DISCONNECT:
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print('Drive synchronized; releasing Colab runtime.')
    runtime.unassign()